# 🎙️ Hệ Thống Lồng Tiếng Video Tiếng Việt Với `k2-fsa/OmniVoice` Trên Google Colab T4

Sổ tay này hỗ trợ **2 cách sử dụng** (bạn chỉ cần chọn **1 trong 2 cách**):

1. **CÁCH 1 (Khuyên dùng — Nhẹ nhất, ổn định nhất):** Dùng Google Colab T4 làm **Máy chủ đọc giọng AI OmniVoice** từ xa. Ứng dụng vẫn mở trên máy tính của bạn (`http://127.0.0.1:7860`), bạn chỉ cần bấm **Play (▶)** ở **Bước 1.1** và **Bước 1.2** bên dưới rồi copy đường link `https://xxxx.trycloudflare.com` dán vào ô **Link Colab T4** trên ứng dụng.
2. **CÁCH 2 (Chạy 100% trên Đám mây Google Colab T4):** Toàn bộ quy trình tải video, tách nhạc, nhận diện chữ, dịch thuật, đọc giọng OmniVoice và ghép video đều chạy trên Google Colab T4, tự động lưu video thành phẩm vào **Google Drive** của bạn (dùng **Bước 2.1** và **Bước 2.2**).

---
## 🚀 CÁCH 1: Biến Colab T4 Thành Máy Chủ Đọc Giọng OmniVoice (Kết nối với App trên máy tính)
*(Hãy đảm bảo bạn đã chọn **Thời gian chạy (Runtime) -> Thay đổi loại thời gian chạy -> T4 GPU**)*

In [ ]:
#@title 🛠️ Bước 1.1: Cài đặt thư viện k2-fsa/OmniVoice & Cloudflare Tunnel (Bấm ▶ chờ khoảng 1–2 phút)
import subprocess, sys, torch

print("🔍 Kiểm tra GPU trên Google Colab...")
if torch.cuda.is_available():
    print(f"✅ Đã nhận GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ CẢNH BÁO: Chưa bật GPU T4! Hãy vào menu Thời gian chạy (Runtime) -> Thay đổi loại thời gian chạy -> Chọn T4 GPU.")

print("📦 Đang cài đặt k2-fsa/OmniVoice, FastAPI và Cloudflared...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "omnivoice", "fastapi", "uvicorn", "soundfile", "torchaudio", "accelerate"], check=True)
subprocess.run(["bash", "-c", "wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared"], check=True)
print("🎉 Cài đặt hoàn tất! Hãy bấm ▶ ở Bước 1.2 ngay bên dưới.")

In [ ]:
#@title 🌐 Bước 1.2: Khởi động Máy chủ OmniVoice & Lấy Link Kết Nối (Giữ ô này chạy)
MODEL_ID = "k2-fsa/OmniVoice" #@param ["k2-fsa/OmniVoice", "splendor1811/omnivoice-vietnamese", "kjanh/KhanhTTS-OmniVoice"]

import asyncio, base64, io, os, re, subprocess, tempfile, threading, time, urllib.request
import numpy as np
import soundfile as sf
import torch
import uvicorn
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from typing import Optional
from omnivoice import OmniVoice

# Dọn cổng 8000 và cloudflared cũ nếu bấm chạy lại
subprocess.run(["bash", "-c", "fuser -k 8000/tcp >/dev/null 2>&1 || true; pkill -f cloudflared >/dev/null 2>&1 || true"])

app = FastAPI(title="OmniVoice Colab T4 Server")
_STATE = globals().get("_OMNI_STATE") or {"model": None, "model_id": None}
globals()["_OMNI_STATE"] = _STATE
_LOCK = threading.Lock()

def get_model(req_model_id: str):
    target = (req_model_id or MODEL_ID).strip() or MODEL_ID
    with _LOCK:
        if _STATE["model"] is None or _STATE["model_id"] != target:
            if _STATE["model"] is not None:
                del _STATE["model"]
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
            dev = "cuda:0" if torch.cuda.is_available() else "cpu"
            dt = torch.float16 if torch.cuda.is_available() else torch.float32
            print(f"⏳ Đang nạp mô hình {target} lên {dev} ({dt})...")
            _STATE["model"] = OmniVoice.from_pretrained(target, device_map=dev, dtype=dt)
            _STATE["model_id"] = target
            print(f"✅ Đã nạp xong mô hình {target}!")
        return _STATE["model"]

class SynthReq(BaseModel):
    text: str
    language: str = "vi"
    ref_audio_b64: Optional[str] = None
    ref_text: Optional[str] = None
    instruct: Optional[str] = None
    duration: Optional[float] = None
    speed: Optional[float] = 1.0
    num_step: int = 32
    model_id: Optional[str] = None

@app.get("/health")
def health():
    dev = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
    return {"ok": True, "device": dev, "model": _STATE["model_id"] or MODEL_ID}

@app.post("/synthesize")
def synthesize(req: SynthReq):
    text = (req.text or "").strip()
    if not text:
        raise HTTPException(status_code=400, detail="Empty text")
    model = get_model(req.model_id or MODEL_ID)
    tmp_ref = None
    try:
        kwargs = {
            "text": text,
            "language": req.language or "vi",
            "num_step": max(8, min(64, int(req.num_step or 32))),
        }
        if req.ref_audio_b64:
            raw = base64.b64decode(req.ref_audio_b64)
            fd, tmp_ref = tempfile.mkstemp(suffix=".wav")
            os.close(fd)
            with open(tmp_ref, "wb") as f:
                f.write(raw)
            kwargs["ref_audio"] = tmp_ref
            if req.ref_text and req.ref_text.strip():
                kwargs["ref_text"] = req.ref_text.strip()
        elif req.instruct and req.instruct.strip():
            kwargs["instruct"] = req.instruct.strip()

        if req.duration and req.duration > 0.3:
            kwargs["duration"] = float(req.duration)
        elif req.speed and abs(req.speed - 1.0) > 0.02:
            kwargs["speed"] = float(req.speed)

        try:
            audio = model.generate(**kwargs)
        except TypeError:
            kwargs.pop("duration", None)
            kwargs.pop("instruct", None)
            audio = model.generate(**kwargs)

        wav = audio[0] if isinstance(audio, (list, tuple)) else audio
        if hasattr(wav, "detach"):
            wav = wav.detach().cpu().numpy()
        wav = np.asarray(wav, dtype=np.float32).squeeze()
        buf = io.BytesIO()
        sf.write(buf, wav, 24000, format="WAV")
        b64 = base64.b64encode(buf.getvalue()).decode("ascii")
        return {"audio_b64": b64, "sr": 24000}
    finally:
        if tmp_ref and os.path.exists(tmp_ref):
            try:
                os.remove(tmp_ref)
            except OSError:
                pass

# 1. Nạp trước mô hình lên GPU T4
get_model(MODEL_ID)

# 2. Khởi chạy Uvicorn trong luồng riêng để không đụng độ vòng lặp asyncio của Google Colab (Python 3.13)
def _start_uvicorn():
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    cfg = uvicorn.Config(app, host="127.0.0.1", port=8000, log_level="warning")
    srv = uvicorn.Server(cfg)
    loop.run_until_complete(srv.serve())

t = threading.Thread(target=_start_uvicorn, daemon=True)
t.start()

# Chờ cổng 8000 sẵn sàng 100%
for _ in range(30):
    try:
        with urllib.request.urlopen("http://127.0.0.1:8000/health", timeout=2) as r:
            if r.status == 200:
                break
    except Exception:
        time.sleep(0.3)

# 3. Mở đường hầm Cloudflare Tunnel nối tới cổng 8000
proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

public_url = None
for _ in range(60):
    line = proc.stdout.readline()
    if not line:
        time.sleep(0.5)
        continue
    m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if m:
        public_url = m.group(0)
        break

print("\n" + "=" * 72)
if public_url:
    print("🎉 MÁY CHỦ OMNIVOICE TRÊN COLAB T4 ĐÃ CHẠY THÀNH CÔNG (CỔNG 8000 OK)!")
    print("👉 HÃY COPY LINK NÀY DÁN VÀO Ô 'Link Colab T4' TRÊN ỨNG DỤNG CỦA BẠN:")
    print(f"\n    {public_url}\n")
else:
    print("⚠️ Không lấy được link Cloudflare tự động, hãy kiểm tra log bên dưới.")
print("=" * 72 + "\n")

# Giữ cho ô Colab luôn chạy để phục vụ yêu cầu đọc giọng từ máy tính của bạn
try:
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    print("🛑 Đã dừng máy chủ OmniVoice.")

---
## ☁️ CÁCH 2: Chạy Trọn Gói 100% Ứng Dụng Lồng Tiếng Ngay Trên Google Colab T4
*(Chỉ dùng Cách 2 nếu bạn không muốn mở ứng dụng trên máy tính cá nhân và muốn làm mọi thứ từ A–Z trên Colab T4, tự động lưu video vào Google Drive)*

In [ ]:
#@title 📦 Bước 2.1: Kết nối Google Drive & Cài đặt Trọn bộ Ứng dụng + OmniVoice
import os, subprocess, sys
from google.colab import drive

print("🔗 Đang kết nối Google Drive để lưu video thành phẩm...")
drive.mount("/content/drive")
OUT_DRIVE = "/content/drive/MyDrive/Douyin_Dubber_Thanh_Pham"
os.makedirs(OUT_DRIVE, exist_ok=True)

REPO_DIR = "/content/video-to-video-change-voice-skill"
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "https://github.com/Ppminh/video-to-video-change-voice-skill.git", REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)

# Nối thư mục THANH_PHAM vào thẳng Google Drive
tp_path = os.path.join(REPO_DIR, "THANH_PHAM")
if os.path.exists(tp_path) and not os.path.islink(tp_path):
    subprocess.run(["rm", "-rf", tp_path])
if not os.path.exists(tp_path):
    os.symlink(OUT_DRIVE, tp_path)

print("⚙️ Đang cài đặt các thư viện xử lý âm thanh, video và OmniVoice...")
subprocess.run(["apt-get", "update", "-qq"], check=True)
subprocess.run(["apt-get", "install", "-y", "-qq", "ffmpeg", "sox", "libsox-fmt-all"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", os.path.join(REPO_DIR, "requirements.txt")], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "omnivoice", "edge-tts", "yt-dlp", "faster-whisper", "demucs", "google-genai", "soundfile", "librosa", "scipy"], check=True)
subprocess.run(["bash", "-c", "wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared"], check=True)
print(f"✅ Hoàn tất! Video làm xong sẽ tự động nằm trong Google Drive: {OUT_DRIVE}")

In [ ]:
#@title ▶️ Bước 2.2: Nhập Gemini API Key & Mở Giao Diện Web Trực Tuyến
GEMINI_API_KEY = "" #@param {type:"string"}
OMNIVOICE_MODEL = "k2-fsa/OmniVoice" #@param ["k2-fsa/OmniVoice", "splendor1811/omnivoice-vietnamese", "kjanh/KhanhTTS-OmniVoice"]

import json, os, re, subprocess, sys, time

REPO_DIR = "/content/video-to-video-change-voice-skill"
cfg_dir = os.path.join(REPO_DIR, "config")
os.makedirs(cfg_dir, exist_ok=True)
if GEMINI_API_KEY.strip():
    with open(os.path.join(cfg_dir, ".env"), "w", encoding="utf-8") as f:
        f.write(f"GEMINI_API_KEY={GEMINI_API_KEY.strip()}\n")

# Đặt mặc định chế độ đọc giọng là OmniVoice trực tiếp trên GPU T4 của Colab
home_cfg = os.path.expanduser("~/.douyin_dubber")
os.makedirs(home_cfg, exist_ok=True)
with open(os.path.join(home_cfg, "settings.json"), "w", encoding="utf-8") as f:
    json.dump({
        "tts_mode": "omnivoice",
        "omnivoice_model": OMNIVOICE_MODEL,
        "omnivoice_url": "",
        "parallel_lanes": False,
        "voice_match_mode": "match_eq"
    }, f, ensure_ascii=False, indent=2)

env = os.environ.copy()
env["PYTHONPATH"] = os.path.join(REPO_DIR, "app")
srv = subprocess.Popen([sys.executable, "-m", "dubber.server"], cwd=REPO_DIR, env=env)
time.sleep(3)

tun = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:7860"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)
web_url = None
for _ in range(60):
    line = tun.stdout.readline()
    if not line:
        time.sleep(0.5)
        continue
    m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if m:
        web_url = m.group(0)
        break

print("\n" + "=" * 72)
if web_url:
    print(f"🌐 BẤM VÀO LINK NÀY ĐỂ MỞ GIAO DIỆN LỒNG TIẾNG TRÊN TRÌNH DUYỆT:")
    print(f"\n    {web_url}\n")
print("=" * 72 + "\n")
srv.wait()